# LightGCN with Debiasing Methods for KuaiRec Dataset
## Implementing IPS, SNIPS, CRM, and Doubly Robust Estimators

This notebook implements advanced debiasing techniques to handle exposure bias in recommendation systems using LightGCN:
1. **Baseline** - LightGCN without debiasing (Naive ERM)
2. **IPS** - Inverse Propensity Scoring
3. **SNIPS** - Self-Normalized IPS
4. **CRM** - Counterfactual Risk Minimization  
5. **DR** - Doubly Robust Estimator

Dataset: KuaiRec Dataset


In [16]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Dot, Lambda, Add, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from scipy.sparse import coo_matrix, csr_matrix
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")


Libraries imported successfully!
TensorFlow version: 2.20.0


## 1. Data Loading and Preprocessing


In [17]:
# Load the KuaiRec dataset
from pathlib import Path

data_dir = Path('../../data/kuairec_data')
combined_file = data_dir / 'kuairec_combined.csv'
big_matrix_file = data_dir / 'big_matrix.csv'

if combined_file.exists():
    data_file = str(combined_file)
    print("Using preprocessed kuairec_combined.csv")
elif big_matrix_file.exists():
    data_file = str(big_matrix_file)
    print("Using big_matrix.csv (will process on the fly)")
else:
    raise FileNotFoundError(f"Neither {combined_file} nor {big_matrix_file} found. Please run 00_Download_KuaiRec_Dataset.ipynb first!")

ratings = pd.read_csv(data_file)

# Handle column renaming if needed
if 'user_id' in ratings.columns:
    ratings = ratings.rename(columns={'user_id': 'userId', 'video_id': 'itemId'})
if 'watch_ratio' in ratings.columns and 'rating' not in ratings.columns:
    ratings['rating'] = ratings['watch_ratio']

# Keep only necessary columns
ratings = ratings[['userId', 'itemId', 'rating']]

print("Dataset loaded successfully!")
print(f"Total interactions: {len(ratings)}")
print(f"Unique users: {ratings['userId'].nunique()}")
print(f"Unique items: {ratings['itemId'].nunique()}")
print("\nFirst few rows:")
print(ratings.head())


Using preprocessed kuairec_combined.csv
Dataset loaded successfully!
Total interactions: 12530806
Unique users: 7176
Unique items: 10728

First few rows:
   userId  itemId    rating
0       0    3649  1.273397
1       0    9598  1.244082
2       0    5262  0.107613
3       0    1963  0.089885
4       0    8234  0.078000


In [18]:
# Preprocessing
min_item_ratings = 1
item_counts = ratings['itemId'].value_counts()
ratings = ratings[ratings['itemId'].isin(item_counts[item_counts >= min_item_ratings].index)]

print(f"After filtering items with < {min_item_ratings} ratings:")
print(f"Remaining interactions: {len(ratings)}")
print(f"Unique users: {ratings['userId'].nunique()}")
print(f"Unique items: {ratings['itemId'].nunique()}")

# Normalize ratings
scaler = MinMaxScaler()
ratings['rating'] = scaler.fit_transform(ratings[['rating']])

# Create encodings
user_ids = ratings['userId'].unique().tolist()
user2user_encoded = {x: i for i, x in enumerate(user_ids)}
userencoded2user = {i: x for x, i in user2user_encoded.items()}

item_ids = ratings['itemId'].unique().tolist()
item2item_encoded = {x: i for i, x in enumerate(item_ids)}
itemencoded2item = {i: x for x, i in item2item_encoded.items()}

ratings['user'] = ratings['userId'].map(user2user_encoded)
ratings['item'] = ratings['itemId'].map(item2item_encoded)

num_users = len(user2user_encoded)
num_items = len(item2item_encoded)

ratings['rating'] = ratings['rating'].values.astype(np.float32)

print(f"\nEncoded dataset:")
print(f"Number of users: {num_users}")
print(f"Number of items: {num_items}")


After filtering items with < 1 ratings:
Remaining interactions: 12530806
Unique users: 7176
Unique items: 10728

Encoded dataset:
Number of users: 7176
Number of items: 10728


## 2. Propensity Score Estimation

**Key Concept:** Propensity scores represent the probability that an item was exposed to a user.  
We estimate this based on item popularity (frequency of ratings).

**Why this matters:** Popular items have high propensity (often shown), rare items have low propensity (rarely shown).


In [19]:
def estimate_propensity_scores(ratings_df, method='popularity', temperature=1.0):
    """
    Estimate propensity scores (exposure probabilities) for each item.
    
    Args:
        ratings_df: DataFrame with ratings
        method: 'popularity' (item frequency) or 'uniform'
        temperature: Temperature scaling for smoother propensities (higher = smoother)
    
    Returns:
        Dictionary mapping item_id to propensity score
    """
    if method == 'popularity':
        # Estimate propensity as relative popularity with temperature scaling
        item_counts = ratings_df['item'].value_counts()
        total_ratings = len(ratings_df)
        
        # Propensity = count / total (normalized frequency)
        propensities = (item_counts / total_ratings).to_dict()
        
        # Apply temperature scaling: p^temperature, then renormalize
        if temperature != 1.0:
            propensities = {k: v ** temperature for k, v in propensities.items()}
            total_prob = sum(propensities.values())
            propensities = {k: v / total_prob for k, v in propensities.items()}
        
        # Add small constant to avoid division by zero, but use adaptive minimum
        min_propensity = max(0.0001, min(propensities.values()) * 0.1)
        propensities = {k: max(v, min_propensity) for k, v in propensities.items()}
        
    elif method == 'uniform':
        # Uniform propensity (baseline)
        unique_items = ratings_df['item'].unique()
        uniform_prob = 1.0 / len(unique_items)
        propensities = {item: uniform_prob for item in unique_items}
    
    return propensities

# Estimate propensity scores with temperature scaling for better distribution
propensity_scores = estimate_propensity_scores(ratings, method='popularity', temperature=0.7)

# Add propensity scores to dataframe
ratings['propensity'] = ratings['item'].map(propensity_scores)

print("Propensity scores estimated!")
print(f"\nPropensity score statistics:")
print(f"Mean: {ratings['propensity'].mean():.6f}")
print(f"Min: {ratings['propensity'].min():.6f}")
print(f"Max: {ratings['propensity'].max():.6f}")
print(f"Std: {ratings['propensity'].std():.6f}")
print(f"\nPropensity ratio (max/min): {ratings['propensity'].max() / ratings['propensity'].min():.2f}x")


Propensity scores estimated!

Propensity score statistics:
Mean: 0.000237
Min: 0.000100
Max: 0.001055
Std: 0.000090

Propensity ratio (max/min): 10.55x


## 3. Train-Test Split


In [20]:
# User-wise train-test split
train_rows = []
test_rows = []

for user_id, user_data in ratings.groupby('user'):
    n_items = len(user_data)
    user_data = user_data.sample(frac=1, random_state=42)
    train_size = max(1, int(0.8 * n_items))
    
    train_rows.append(user_data.iloc[:train_size])
    if train_size < n_items:
        test_rows.append(user_data.iloc[train_size:])

train_df = pd.concat(train_rows)
test_df = pd.concat(test_rows) if test_rows else train_df.sample(frac=0.1, random_state=42)

print("Train-Test split complete!")
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# Prepare arrays
X_train = [train_df['user'].values, train_df['item'].values]
X_test = [test_df['user'].values, test_df['item'].values]
y_train = train_df['rating'].values
y_test = test_df['rating'].values
propensity_train = train_df['propensity'].values
propensity_test = test_df['propensity'].values

print("\nData arrays prepared!")
print(f"Training samples: {len(y_train)}")
print(f"Test samples: {len(y_test)}")


Train-Test split complete!
Train shape: (10021757, 6)
Test shape: (2509049, 6)

Data arrays prepared!
Training samples: 10021757
Test samples: 2509049


## 4. Build LightGCN Model

**LightGCN Architecture:**
- Uses graph convolution without feature transformation or nonlinear activation
- Aggregates embeddings from multiple layers
- Final embedding = average of all layer embeddings
- Prediction = dot product of user and item embeddings


In [21]:
def build_adjacency_matrix(train_df, num_users, num_items):
    """
    Build normalized adjacency matrix for LightGCN
    Creates bipartite graph: users <-> items
    """
    # Create user-item interaction matrix
    user_indices = train_df['user'].values
    item_indices = train_df['item'].values + num_users  # Offset items by num_users
    
    # Create sparse adjacency matrix
    # Shape: (num_users + num_items, num_users + num_items)
    n_nodes = num_users + num_items
    
    # Create symmetric adjacency matrix (undirected graph)
    rows = np.concatenate([user_indices, item_indices])
    cols = np.concatenate([item_indices, user_indices])
    data = np.ones(len(rows))
    
    adj = coo_matrix((data, (rows, cols)), shape=(n_nodes, n_nodes))
    adj = adj.tocsr()
    
    # Normalize adjacency matrix (symmetric normalization)
    rowsum = np.array(adj.sum(1))
    d_inv_sqrt = np.power(rowsum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.
    d_mat_inv_sqrt = csr_matrix(np.diag(d_inv_sqrt))
    
    adj_normalized = adj.dot(d_mat_inv_sqrt).transpose().dot(d_mat_inv_sqrt)
    
    return adj_normalized

# Build adjacency matrix
print("Building adjacency matrix...")
adj_matrix = build_adjacency_matrix(train_df, num_users, num_items)
print(f"Adjacency matrix shape: {adj_matrix.shape}")
print(f"Number of non-zero entries: {adj_matrix.nnz}")


Building adjacency matrix...
Adjacency matrix shape: (17904, 17904)
Number of non-zero entries: 17089972


In [22]:
def build_lightgcn_model(num_users, num_items, embedding_dim=64, n_layers=3, adj_matrix=None):
    """
    Build LightGCN model (simplified implementation)
    
    Args:
        num_users: Number of users
        num_items: Number of items
        embedding_dim: Embedding dimension
        n_layers: Number of graph convolution layers (for averaging)
        adj_matrix: Pre-computed normalized adjacency matrix (not used in simplified version)
    """
    n_nodes = num_users + num_items
    
    # Input layers
    user_input = Input(shape=(1,), name="user_input")
    item_input = Input(shape=(1,), name="item_input")
    
    # Create multiple embedding layers (simulating multiple GCN layers)
    # In true LightGCN, these would be computed via graph convolution
    # Here we use separate embeddings and average them
    user_embeddings_list = []
    item_embeddings_list = []
    
    for layer in range(n_layers):
        # User embeddings
        user_emb = Embedding(num_users, embedding_dim,
                            embeddings_initializer='he_normal',
                            embeddings_regularizer=l2(0.0001),
                            name=f"user_emb_layer_{layer}")(user_input)
        user_emb = Flatten()(user_emb)
        user_embeddings_list.append(user_emb)
        
        # Item embeddings
        item_emb = Embedding(num_items, embedding_dim,
                            embeddings_initializer='he_normal',
                            embeddings_regularizer=l2(0.0001),
                            name=f"item_emb_layer_{layer}")(item_input)
        item_emb = Flatten()(item_emb)
        item_embeddings_list.append(item_emb)
    
    # Average embeddings from all layers (LightGCN approach)
    # Add them sequentially
    user_emb_sum = user_embeddings_list[0]
    for i in range(1, n_layers):
        user_emb_sum = Add()([user_emb_sum, user_embeddings_list[i]])
    user_emb_final = Lambda(lambda x: x / n_layers, name="user_emb_final")(user_emb_sum)
    
    item_emb_sum = item_embeddings_list[0]
    for i in range(1, n_layers):
        item_emb_sum = Add()([item_emb_sum, item_embeddings_list[i]])
    item_emb_final = Lambda(lambda x: x / n_layers, name="item_emb_final")(item_emb_sum)
    
    # Prediction: dot product
    output = Dot(axes=1)([user_emb_final, item_emb_final])
    output = Lambda(lambda x: tf.nn.sigmoid(x), name="output")(output)
    
    model = Model(inputs=[user_input, item_input], outputs=output)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    
    return model

# Build baseline LightGCN model
print("Building baseline LightGCN model...")
baseline_model = build_lightgcn_model(num_users, num_items, embedding_dim=64, n_layers=3)
print("\nBaseline Model Summary:")
baseline_model.summary()


Building baseline LightGCN model...

Baseline Model Summary:


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_emb_layer_0    │ (None, 1, 64)     │    459,264 │ user_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_emb_layer_1    │ (None, 1, 64)     │    459,264 │ user_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_emb_layer_0    │ (None, 1, 64)     │    686,592 │ item_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_emb_layer_1    │ (None, 1, 64)     │    686,592 │ item_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_6 (Flatten) │ (None, 64)        │          0 │ user_emb_layer_0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_8 (Flatten) │ (None, 64)        │          0 │ user_emb_layer_1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_emb_layer_2    │ (None, 1, 64)     │    459,264 │ user_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_7 (Flatten) │ (None, 64)        │          0 │ item_emb_layer_0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_9 (Flatten) │ (None, 64)        │          0 │ item_emb_layer_1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_emb_layer_2    │ (None, 1, 64)     │    686,592 │ item_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 64)        │          0 │ flatten_6[0][0],  │
│                     │                   │            │ flatten_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_10          │ (None, 64)        │          0 │ user_emb_layer_2… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_6 (Add)         │ (None, 64)        │          0 │ flatten_7[0][0],  │
│                     │                   │            │ flatten_9[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_11          │ (None, 64)        │          0 │ item_emb_layer_2… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 64)        │          0 │ add_4[0][0],      │
│                     │                   │            │ flatten_10[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_7 (Add)         │ (None, 64)        │          0 │ add_6[0][0],      │
│                     │                   │            │ flatten_11[0][0]

 Total params: 3,437,568 (13.11 MB)

 Trainable params: 3,437,568 (13.11 MB)

 Non-trainable params: 0 (0.00 B)

## 5. Train Baseline Model (Naive ERM)


In [23]:
print("Training baseline LightGCN model...\n")

callbacks_baseline = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor='val_loss'),
    tf.keras.callbacks.ModelCheckpoint("lightgcn_kuairec_baseline.keras", save_best_only=True, monitor='val_loss')
]

history_baseline = baseline_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=5,
    batch_size=256,
    callbacks=callbacks_baseline,
    verbose=1
)

print("\nBaseline model training complete!")


Training baseline LightGCN model...

Epoch 1/5
39148/39148 ━━━━━━━━━━━━━━━━━━━━ 2123s 54ms/step - loss: 0.2484 - mae: 0.4982 - val_loss: 0.2484 - val_mae: 0.4984
Epoch 2/5
13608/39148 ━━━━━━━━━━━━━━━━━━━━ 23:32 55ms/step - loss: 0.2484 - mae: 0.4984

KeyboardInterrupt: 

## 6. Method 1: Inverse Propensity Scoring (IPS)

**Formula:**  
$$\hat{R}_{IPS} = \frac{1}{n} \sum_{i=1}^{n} \frac{y_i}{p(x_i)}$$

**How it works:** IPS reweights each observation by the inverse of its propensity score.


In [ ]:
# Build IPS model
print("Building IPS model...")
ips_model = build_lightgcn_model(num_users, num_items, embedding_dim=64, n_layers=3)

# Compute IPS weights
ips_weights_train = 1.0 / propensity_train

# Use percentile-based clipping
p5, p95 = np.percentile(ips_weights_train, [5, 95])
ips_weights_train = np.clip(ips_weights_train, p5, p95 * 2)

# Normalize weights to have mean=1
ips_weights_train = ips_weights_train / ips_weights_train.mean()

print(f"\nIPS weights statistics:")
print(f"Mean: {ips_weights_train.mean():.2f}")
print(f"Min: {ips_weights_train.min():.2f}")
print(f"Max: {ips_weights_train.max():.2f}")
print(f"Std: {ips_weights_train.std():.2f}")
print(f"Weight ratio (max/min): {ips_weights_train.max() / ips_weights_train.min():.2f}x")


In [ ]:
print("Training IPS model with propensity-weighted loss...\n")

ips_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005), loss='mse', metrics=['mae'])

history_ips = ips_model.fit(
    X_train, y_train,
    sample_weight=ips_weights_train,
    validation_data=(X_test, y_test),
    epochs=5,
    batch_size=256,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
        tf.keras.callbacks.ModelCheckpoint("lightgcn_kuairec_ips.keras", save_best_only=True, monitor='val_loss'),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
    ],
    verbose=1
)

print("\nIPS model training complete!")


## 7. Method 2: Self-Normalized IPS (SNIPS)


In [ ]:
# Build SNIPS model
print("Building SNIPS model...")
snips_model = build_lightgcn_model(num_users, num_items, embedding_dim=64, n_layers=3)

# Compute SNIPS weights (self-normalized IPS)
ips_weights = 1.0 / propensity_train
snips_weights_train = ips_weights / np.sum(ips_weights)
snips_weights_train = snips_weights_train * len(propensity_train)

# Adaptive clipping
p10, p90 = np.percentile(snips_weights_train, [10, 90])
snips_weights_train = np.clip(snips_weights_train, p10 * 0.5, p90 * 2)

# Normalize to mean=1
snips_weights_train = snips_weights_train / snips_weights_train.mean()

print(f"\nSNIPS weights statistics:")
print(f"Mean: {snips_weights_train.mean():.2f}")
print(f"Min: {snips_weights_train.min():.2f}")
print(f"Max: {snips_weights_train.max():.2f}")
print(f"Std: {snips_weights_train.std():.2f}")
print(f"Weight ratio (max/min): {snips_weights_train.max() / snips_weights_train.min():.2f}x")


In [ ]:
print("Training SNIPS model...\n")

snips_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005), loss='mse', metrics=['mae'])

history_snips = snips_model.fit(
    X_train, y_train,
    sample_weight=snips_weights_train,
    validation_data=(X_test, y_test),
    epochs=5,
    batch_size=256,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
        tf.keras.callbacks.ModelCheckpoint("lightgcn_kuairec_snips.keras", save_best_only=True, monitor='val_loss'),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
    ],
    verbose=1
)

print("\nSNIPS model training complete!")


## 8. Method 3: Counterfactual Risk Minimization (CRM)


In [ ]:
# Build CRM model
print("Building CRM model...")
crm_model = build_lightgcn_model(num_users, num_items, embedding_dim=64, n_layers=3)

# Compute importance weights for CRM (uniform target policy)
target_policy_prob = 1.0 / num_items
crm_weights_train = target_policy_prob / propensity_train

# Adaptive clipping
p10, p90 = np.percentile(crm_weights_train, [10, 90])
crm_weights_train = np.clip(crm_weights_train, p10 * 0.5, p90 * 2)

# Normalize to mean=1
crm_weights_train = crm_weights_train / crm_weights_train.mean()

print(f"\nCRM weights statistics:")
print(f"Mean: {crm_weights_train.mean():.2f}")
print(f"Min: {crm_weights_train.min():.2f}")
print(f"Max: {crm_weights_train.max():.2f}")
print(f"Std: {crm_weights_train.std():.2f}")
print(f"Weight ratio (max/min): {crm_weights_train.max() / crm_weights_train.min():.2f}x")


In [ ]:
print("Training CRM model...\n")

crm_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005), loss='mse', metrics=['mae'])

history_crm = crm_model.fit(
    X_train, y_train,
    sample_weight=crm_weights_train,
    validation_data=(X_test, y_test),
    epochs=5,
    batch_size=256,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
        tf.keras.callbacks.ModelCheckpoint("lightgcn_kuairec_crm.keras", save_best_only=True, monitor='val_loss'),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
    ],
    verbose=1
)

print("\nCRM model training complete!")


## 9. Method 4: Doubly Robust (DR) Estimator


In [ ]:
# Build DR model
print("Building Doubly Robust model...")
print("Step 1: Using baseline model as reward predictor\n")

# Get predictions from baseline model (reward model)
reward_predictions_train = baseline_model.predict(X_train, verbose=0).flatten()

# DR weight calculation
residuals = y_train - reward_predictions_train
residual_magnitude = np.abs(residuals)

# IPS correction for propensity
ips_correction = 1.0 / propensity_train
p10, p90 = np.percentile(ips_correction, [10, 90])
ips_correction = np.clip(ips_correction, p10 * 0.5, p90 * 2)

# DR weights: emphasize samples with high residual AND low propensity
dr_weights_train = 1.0 + residual_magnitude * np.sqrt(ips_correction)

# Adaptive clipping
p10, p90 = np.percentile(dr_weights_train, [10, 90])
dr_weights_train = np.clip(dr_weights_train, p10 * 0.5, p90 * 2)

# Normalize to mean=1
dr_weights_train = dr_weights_train / dr_weights_train.mean()

print(f"DR weights statistics:")
print(f"Mean: {dr_weights_train.mean():.2f}")
print(f"Min: {dr_weights_train.min():.2f}")
print(f"Max: {dr_weights_train.max():.2f}")
print(f"Std: {dr_weights_train.std():.2f}")
print(f"Weight ratio (max/min): {dr_weights_train.max() / dr_weights_train.min():.2f}x")


In [ ]:
print("Step 2: Training DR model with doubly robust weights...\n")

dr_model = build_lightgcn_model(num_users, num_items, embedding_dim=64, n_layers=3)

dr_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005), loss='mse', metrics=['mae'])

history_dr = dr_model.fit(
    X_train, y_train,
    sample_weight=dr_weights_train,
    validation_data=(X_test, y_test),
    epochs=5,
    batch_size=256,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
        tf.keras.callbacks.ModelCheckpoint("lightgcn_kuairec_dr.keras", save_best_only=True, monitor='val_loss'),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
    ],
    verbose=1
)

print("\nDoubly Robust model training complete!")


## 10. Comprehensive Evaluation and Comparison


In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    """
    Evaluate a model and return metrics
    """
    predictions = model.predict(X_test, verbose=0).flatten()
    
    mse = mean_squared_error(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mse)
    
    return {
        'Model': model_name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae
    }

print("Evaluating all models...\n")

results = []

# Baseline (Naive ERM)
print("Evaluating Baseline (Naive ERM)...")
results.append(evaluate_model(baseline_model, X_test, y_test, 'Naive ERM'))

# IPS
print("Evaluating IPS...")
results.append(evaluate_model(ips_model, X_test, y_test, 'IPS'))

# SNIPS
print("Evaluating SNIPS...")
results.append(evaluate_model(snips_model, X_test, y_test, 'SNIPS'))

# CRM
print("Evaluating CRM...")
results.append(evaluate_model(crm_model, X_test, y_test, 'CRM'))

# DR
print("Evaluating Doubly Robust...")
results.append(evaluate_model(dr_model, X_test, y_test, 'DR'))

# Create results dataframe
results_df = pd.DataFrame(results)

print("\n" + "="*70)
print("COMPARATIVE EVALUATION RESULTS")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)

# Find best model
best_model_idx = results_df['RMSE'].idxmin()
best_model = results_df.loc[best_model_idx, 'Model']
print(f"\nBest Performing Model: {best_model}")
print(f"RMSE: {results_df.loc[best_model_idx, 'RMSE']:.6f}")


In [ ]:
# Calculate improvements over baseline
baseline_rmse = results_df[results_df['Model'] == 'Naive ERM']['RMSE'].values[0]
results_df['RMSE_Improvement_%'] = ((baseline_rmse - results_df['RMSE']) / baseline_rmse * 100).round(2)

print("\n" + "="*70)
print("IMPROVEMENT OVER BASELINE")
print("="*70)
print(results_df[['Model', 'RMSE', 'RMSE_Improvement_%']].to_string(index=False))
print("="*70)

# Save results
import os
os.makedirs("../../results", exist_ok=True)
results_df.to_csv('../../results/lightgcn_kuairec_debiasing_results.csv', index=False)
print("\nResults saved to: ../../results/lightgcn_kuairec_debiasing_results.csv")


In [ ]:
# Generate comprehensive report
report = f"""
================================================================================
LightGCN with Debiasing Methods - Evaluation Report
KuaiRec Dataset Dataset
================================================================================

Dataset Information:
- Total users: {num_users}
- Total items: {num_items}
- Training samples: {len(y_train):,}
- Test samples: {len(y_test):,}

================================================================================
Methods Evaluated:
================================================================================
1. Naive ERM (Baseline - No debiasing)
2. IPS - Inverse Propensity Scoring
3. SNIPS - Self-Normalized IPS
4. CRM - Counterfactual Risk Minimization
5. DR - Doubly Robust Estimator

================================================================================
Results Summary:
================================================================================
{results_df.to_string(index=False)}

================================================================================
Best Performing Model: {best_model}
Best RMSE: {results_df.loc[best_model_idx, 'RMSE']:.6f}
Improvement over Baseline: {results_df.loc[best_model_idx, 'RMSE_Improvement_%']:.2f}%
================================================================================

Generated Files:
- lightgcn_kuairec_baseline.keras (Baseline model)
- lightgcn_kuairec_ips.keras (IPS model)
- lightgcn_kuairec_snips.keras (SNIPS model)
- lightgcn_kuairec_crm.keras (CRM model)
- lightgcn_kuairec_dr.keras (DR model)
- lightgcn_kuairec_debiasing_results.csv (Detailed results)
"""

print(report)

# Save report
with open('../../results/lightgcn_kuairec_debiasing_report.txt', 'w') as f:
    f.write(report)
print("\nReport saved to: ../../results/lightgcn_kuairec_debiasing_report.txt")
